# Gate features

One row per slate (not per candidate). They feed the gate `w = sigmoid(a·log1p(history_length) + b·log1p(time_since_last_activity) + c·log1p(purchase_count) + bias)` that mixes Model 1 (short-term) and Model 2 (long-term): `score = w·P1 + (1-w)·P2`. For very short histories the gate is overridden to `w = 1` (Model 1 only).

**Inputs:** `data/train_sampled.parquet`, `val.parquet`, `test.parquet` (only the history and candidate columns are read).
**Outputs:** `features/gate_train.parquet`, `gate_val.parquet`, `gate_test.parquet`.

**History filter:** none (`FILTER_MODE = "none"`), the same setting as the Model 1 feature notebook. The filter code is kept so a sensitivity check can be rerun; keep the setting identical in both notebooks.

## 0. Setup

In [16]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/Grp8Submission")
except ImportError:              # not on Colab
    PROJECT = Path(".")
DATA = PROJECT / "data"
FEAT_M1 = PROJECT / "features" / "model1"
FEAT_M2 = PROJECT / "features" / "model2"
FEAT_GATE = PROJECT / "features" / "gate"

for d in (DATA, FEAT_M1, FEAT_M2):
    d.mkdir(parents=True, exist_ok=True)
OUT = FEAT_GATE

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
import os
import numpy as np, pandas as pd, pyarrow.parquet as pq
from pathlib import Path
from collections import Counter, defaultdict
from sklearn.metrics import roc_auc_score

# split name -> parquet file name (without .parquet).
SPLITS = {"train": "train_sampled", "holdout": "val1", "val": "val2", "test": "test"}



FILTER_MODE = "none"       # must match the Model 1 feature notebook ("none" | "in_slate" | "all")
MIN_AGE_DAYS = 0.05        # only used when FILTER_MODE != "none"
SHORT_HISTORY_PCTL = 10    # hard override w=1 below this train percentile of history length
N_CV_FOLDS = 5
SEED = 0

# candidate_wid_list is only needed for the in_slate filter
HIST = ["slate_id", "candidate_wid_list", "history_wid_list", "history_type_list", "history_days_before"]

# Setup: on Colab, mount Google Drive (same project folder in every notebook).
# Locally, use the repo folder (data/ and features/ next to this notebook).


def load(path, columns=None):
    # ignore_metadata avoids the pandas 2.x/3.x pyarrow-metadata TypeError
    return pq.read_table(path, columns=columns).to_pandas(ignore_metadata=True)


def slate_keys(df):
    """One int64 key per (slate, candidate): slate_id * 2**32 + wid."""
    lens = df["candidate_wid_list"].map(len).to_numpy()
    sid = np.repeat(df["slate_id"].to_numpy().astype(np.int64), lens)
    wid = np.concatenate(df["candidate_wid_list"].to_numpy()).astype(np.int64)
    return sid * (1 << 32) + wid


def clean_history(h, keys, mode=None, min_age=None):
    mode = FILTER_MODE if mode is None else mode
    min_age = MIN_AGE_DAYS if min_age is None else min_age
    if mode == "none":
        return h
    recent = h["days"].to_numpy() < min_age
    if mode == "all":
        return h[~recent]
    if mode == "in_slate":
        hk = h["slate_id"].to_numpy().astype(np.int64) * (1 << 32) + h["wid"].to_numpy()
        return h[~(recent & np.isin(hk, keys))]
    raise ValueError("mode must be 'in_slate', 'all' or 'none'")

## 1. Gate feature definitions
- `gate_history_length`: number of history events (all types) left after the history filter (with `FILTER_MODE = "none"`: all events).
- `gate_time_since_last_activity`: days since the most recent such event. Empty history -> filled with the maximum observed age (365) and flagged.
- `gate_purchase_count`: number of ORD events in history. Orders are observed over the full year, so unlike clicks and carts this count is comparable across users.
- `log1p` versions are what enter the sigmoid (long right tails).

In [18]:
def build_gate_features(df, keys=None, mode=None, min_age=None):
    n_rows = len(df)
    lens = df["history_wid_list"].map(len).to_numpy()
    h = pd.DataFrame({
        "slate_id": np.repeat(df["slate_id"].to_numpy(), lens),
        "row": np.repeat(np.arange(n_rows), lens),
        "wid": np.concatenate(df["history_wid_list"].to_numpy()).astype(np.int64),
        "typ": np.concatenate(df["history_type_list"].to_numpy()).astype(str),
        "days": np.concatenate(df["history_days_before"].to_numpy()).astype(np.float32),   # float32, same as Model 1
    })
    mode_ = FILTER_MODE if mode is None else mode
    if mode_ != "none" and keys is None:
        keys = slate_keys(df)
    h = clean_history(h, keys, mode, min_age)
    row, typ, days = h["row"].to_numpy(), h["typ"].to_numpy(), h["days"].to_numpy().astype(np.float64)

    length = np.bincount(row, minlength=n_rows)
    purchases = np.bincount(row[typ == "ORD"], minlength=n_rows)
    last = pd.Series(days).groupby(row).min().reindex(range(n_rows))       # NaN when no events left
    empty = length == 0
    last = last.fillna(365.0).to_numpy()

    out = pd.DataFrame({
        "slate_id": df["slate_id"].to_numpy(),
        "gate_history_length": length.astype(np.int32),
        "gate_time_since_last_activity": last.astype(np.float32),
        "gate_purchase_count": purchases.astype(np.int32),
        "gate_empty_history": empty.astype(np.int8),
    })
    out["gate_log_history_length"] = np.log1p(out["gate_history_length"]).astype(np.float32)
    out["gate_log_time_since_last"] = np.log1p(out["gate_time_since_last_activity"]).astype(np.float32)
    out["gate_log_purchase_count"] = np.log1p(out["gate_purchase_count"]).astype(np.float32)
    return out

## 2. Build for all splits

In [19]:
gate = {}
for name, fn in SPLITS.items():
    d = load(f"{DATA}/{fn}.parquet", columns=HIST)
    gate[name] = build_gate_features(d)
    print(name, gate[name].shape)
    del d

train (122083, 8)
holdout (17799, 8)
val (16924, 8)
test (17025, 8)


## 3. Short-history threshold, standardisation constants, CV folds
Threshold and standardisation use **train only**. `gate_cv_fold` (0-4, by slate) is added to `val` only.

In [23]:
LOGS = ["gate_log_history_length", "gate_log_time_since_last", "gate_log_purchase_count"]

thr = int(np.percentile(gate["train"]["gate_history_length"], SHORT_HISTORY_PCTL))
print("short-history threshold (train 10th percentile):", thr)

mu = gate["train"][LOGS].mean()
sd = gate["train"][LOGS].std()
scaler = pd.DataFrame({"mean": mu, "std": sd})
scaler.to_csv(f"{OUT}/gate_scaler.csv")
print(scaler.round(4))

for name, g in gate.items():
    g["gate_short_history"] = (g["gate_history_length"] <= thr).astype(np.int8)
    for c in LOGS:
        g[c.replace("gate_log_", "gate_z_")] = ((g[c] - mu[c]) / sd[c]).astype(np.float32)

rng = np.random.default_rng(SEED)
gate["val"]["gate_cv_fold"] = rng.permutation(np.arange(len(gate["val"])) % N_CV_FOLDS).astype(np.int8)

for name, g in gate.items():
    g.to_parquet(f"{OUT}/gate_{name}.parquet", index=False)
    print(name, "override share:", round(g["gate_short_history"].mean() * 100, 2), "% | empty:", int(g["gate_empty_history"].sum()))
print("val CV fold sizes:", gate["val"]["gate_cv_fold"].value_counts().sort_index().tolist())

short-history threshold (train 10th percentile): 10
                            mean     std
gate_log_history_length   4.2658  1.3521
gate_log_time_since_last  0.7608  1.0988
gate_log_purchase_count   2.6946  1.4992
train override share: 10.3 % | empty: 0
holdout override share: 11.17 % | empty: 0
val override share: 10.23 % | empty: 0
test override share: 10.23 % | empty: 0
val CV fold sizes: [3385, 3385, 3385, 3385, 3384]


## 4. Descriptive statistics

In [24]:
cols = ["gate_history_length", "gate_time_since_last_activity", "gate_purchase_count"]
for name, g in gate.items():
    print("=====", name)
    print(g[cols].describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(3).to_string())

===== train
       gate_history_length  gate_time_since_last_activity  gate_purchase_count
count           122083.000                     122083.000           122083.000
mean               153.341                          5.431               39.144
std                224.121                         22.001               69.764
min                  1.000                          0.000                0.000
10%                 10.000                          0.000                1.000
25%                 29.000                          0.001                4.000
50%                 78.000                          0.222               15.000
75%                188.000                          2.139               44.000
90%                375.000                          9.658              103.000
99%               1059.180                         89.301              326.000
max               4802.000                        363.959             2899.000
===== holdout
       gate_history_length

## 5. Sanity checks

In [26]:
for name, g in gate.items():
    assert g["slate_id"].is_unique
    assert (g["gate_history_length"] >= 0).all() and (g["gate_purchase_count"] <= g["gate_history_length"]).all()
    assert g["gate_time_since_last_activity"].between(0, 366).all()
    assert not g.isna().any().any()
print("basic checks OK")

# brute-force check on a small sample (val), written independently of clean_history
d = load(f"{DATA}/val2.parquet", columns=HIST).head(200)
g = gate["val"].set_index("slate_id")
thr_age = np.float32(MIN_AGE_DAYS)
def keep(wid, day, cands):
    if FILTER_MODE == "none":
        return True
    recent = np.float32(day) < thr_age
    if FILTER_MODE == "all":
        return not recent
    return not (recent and int(wid) in cands)
for sid, cands, wids, types, days in zip(d["slate_id"], d["candidate_wid_list"], d["history_wid_list"],
                                         d["history_type_list"], d["history_days_before"]):
    cs = set(int(c) for c in cands)
    k = [(t, np.float32(x)) for w, t, x in zip(wids, types, days) if keep(w, x, cs)]
    assert g.loc[sid, "gate_history_length"] == len(k)
    assert g.loc[sid, "gate_purchase_count"] == sum(t == "ORD" for t, _ in k)
    if k:
        assert abs(g.loc[sid, "gate_time_since_last_activity"] - min(x for _, x in k)) < 1e-4
print("brute-force check on 200 val slates OK")

# train vs val vs test distribution (they should look alike)
print(pd.DataFrame({n: g[["gate_log_history_length", "gate_log_time_since_last", "gate_log_purchase_count"]].mean()
                    for n, g in gate.items()}).round(3))

basic checks OK
brute-force check on 200 val slates OK
                          train  holdout    val   test
gate_log_history_length   4.266    4.230  4.293  4.265
gate_log_time_since_last  0.761    0.772  0.753  0.751
gate_log_purchase_count   2.695    2.658  2.717  2.697


### 5.1 Is `time since last activity` still informative?
With no history filter, most searches have an event just before the query. If nearly all values sit at almost zero, `log1p` of them is almost constant and the gate cannot use this feature.

In [27]:
t = gate["train"]["gate_time_since_last_activity"]
print("share of slates with last activity < 1 min / < 15 min / < 1 h / < 1 d:",
      [round(float((t < x).mean()), 3) for x in (1/1440, 15/1440, 1/24, 1)])
print(t.describe(percentiles=[.1, .25, .5, .75, .9]).round(4).to_string())
print("std of log time since last (train):", round(float(gate["train"]["gate_log_time_since_last"].std()), 3))

share of slates with last activity < 1 min / < 15 min / < 1 h / < 1 d: [0.23, 0.397, 0.433, 0.642]
count    122083.0000
mean          5.4313
std          22.0013
min           0.0000
10%           0.0002
25%           0.0008
50%           0.2221
75%           2.1387
90%           9.6579
max         363.9587
std of log time since last (train): 1.099


### 5.2 Collinearity between the log gate features
The logistic gate is sensitive to collinear inputs. VIF below about 5 is fine.

In [28]:
X = gate["train"][["gate_log_history_length", "gate_log_time_since_last", "gate_log_purchase_count"]]
corr = X.corr()
print(corr.round(2).to_string())
vif = pd.Series(np.diag(np.linalg.inv(corr.to_numpy())), index=corr.index)
print("\nVIF:", vif.round(2).to_dict())

                          gate_log_history_length  gate_log_time_since_last  gate_log_purchase_count
gate_log_history_length                      1.00                     -0.46                     0.75
gate_log_time_since_last                    -0.46                      1.00                    -0.19
gate_log_purchase_count                      0.75                     -0.19                     1.00

VIF: {'gate_log_history_length': 2.99, 'gate_log_time_since_last': 1.36, 'gate_log_purchase_count': 2.45}


## 6. Handover: what the gate feature step provides and what comes next

**What this notebook provides** (all one row per slate, keyed by `slate_id`):
- `features/gate_val.parquet` (fit the gate here), `gate_test.parquet` (final evaluation), plus `gate_train.parquet` and `gate_holdout.parquet` (not used for fitting).
- Columns: raw (`gate_history_length`, `gate_time_since_last_activity`, `gate_purchase_count`), log versions, standardised versions `gate_z_*` (use these as the gate inputs), `gate_short_history` (1 means the gate is overridden to `w = 1`), `gate_empty_history`.
- `gate_val` also has `gate_cv_fold` (0-4, assigned by slate) for the cross-fit.
- `gate_scaler.csv`: train mean and std behind the `gate_z_*` columns. The short-history threshold is printed in section 3. Both come from train only, and the same constants are applied to val and test.

**What the gate trainer needs to bring:**
1. `data/val_scores.parquet` and `data/test_scores.parquet`: one row per (slate, candidate) with `slate_id`, `label`, `p1`, `p2`. P1 and P2 come from the final Model 1 and Model 2 (fixed after tuning on holdout), scored on every val and test candidate (full slates, no negative sampling).
2. Decide the binary target once. Platt scaling and the gate loss currently use `label > 0` (any interaction).

**Next steps:**
1. Join the gate features to the candidate rows by `slate_id`.
2. Fit Platt scaling for P1 and P2 on val, separately per model.
3. Fit the gate on val: minimise log loss of `w·P1 + (1-w)·P2`, with `w = sigmoid(a·z_len + b·z_time + c·z_purchase + bias)` and `w = 1` where `gate_short_history == 1`.
4. For reporting, use the 5-fold cross-fit by `gate_cv_fold` (section 7). Fit calibration and gate on 4 folds, score the fifth, average.
5. Fit the final gate on all of val, then evaluate once on test: Model 1 alone, Model 2 alone, the blend and the baselines. Never refit on test.
6. Optional evidence table: Model 1 NDCG@10 by gate bucket (history length, purchase count, time since last activity), to show why a gate is needed.
